# 🚀 ETF Portfolio Backtesting System - Integrated System (All-in-One)

**Run ครั้งเดียว - ใช้ได้ทุกอย่าง!**

## 📋 วิธีใช้งาน:
1. **แก้ password ใน Cell 1** (ด้านล่าง)
2. **Run Cell 1:** Setup & Load Functions
3. **Run Cell 2:** Start Main Controller (ใช้งานไปเรื่อยๆ)
4. **ใช้ Menu ตามต้องการ** จนกว่าจะเลือก 0 (Exit)

---

## 🎯 Main Controller จะ:
- แสดง Menu
- รอรับคำสั่ง
- เรียก Module ที่เหมาะสม
- แสดงผลลัพธ์
- กลับไป Menu
- วนไปเรื่อยๆ จนกว่าคุณจะกด 0 (Exit)

---

# Cell 1: Setup & Load Everything

⚠️ **แก้ password ตรงนี้แล้ว Run cell นี้**

In [ ]:
# ===================================================================
# CONFIGURATION - แก้ตรงนี้!
# ===================================================================

DB_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',  # ⚠️ แก้ตรงนี้!
    'database': 'etf_backtesting'
}

# ===================================================================
# IMPORTS & SETUP
# ===================================================================

import os
import sys
from pathlib import Path
import importlib.util
import mysql.connector
import pandas as pd
from IPython.display import clear_output

PROJECT_DIR = Path(os.getcwd()).absolute()

# ===================================================================
# MAIN CONTROLLER CLASS
# ===================================================================

class IntegratedSystem:
    """Main Controller - Integrated System"""
    
    def __init__(self, db_config, project_dir):
        self.db_config = db_config
        self.project_dir = project_dir
        self.modules = {}
        self.running = True
    
    def load_module(self, module_name, file_path):
        """Load module dynamically"""
        try:
            full_path = self.project_dir / file_path
            if not full_path.exists():
                print(f"⚠️  {file_path} not found")
                return False
            
            spec = importlib.util.spec_from_file_location(module_name, full_path)
            module = importlib.util.module_from_spec(spec)
            sys.modules[module_name] = module
            spec.loader.exec_module(module)
            self.modules[module_name] = module
            print(f"✓ {module_name}")
            return True
        except Exception as e:
            print(f"✗ {module_name}: {e}")
            return False
    
    def initialize(self):
        """Initialize system"""
        print("\n" + "="*80)
        print("🚀 INITIALIZING INTEGRATED SYSTEM")
        print("="*80)
        
        # Test connection
        print("\n🔌 Testing MySQL connection...")
        try:
            conn = mysql.connector.connect(**self.db_config)
            cursor = conn.cursor()
            cursor.execute("SELECT VERSION()")
            version = cursor.fetchone()[0]
            cursor.close()
            conn.close()
            print(f"✅ MySQL Connected (Version: {version})")
        except Exception as e:
            print(f"❌ Connection Failed: {e}")
            return False
        
        # Load modules
        print("\n📦 Loading Modules...")
        modules_to_load = [
            ('crud_operations', 'crud_operations/crud_operations.py'),
            ('backtesting_engine', 'backtesting/backtesting_engine.py'),
            ('analytics', 'analytics/analytics.py'),
        ]
        
        for name, path in modules_to_load:
            self.load_module(name, path)
        
        print("\n" + "="*80)
        print("✅ System Ready!")
        print("="*80)
        return True
    
    def get_connection(self):
        return mysql.connector.connect(**self.db_config)
    
    # ================================================================
    # PORTFOLIO FUNCTIONS
    # ================================================================
    
    def view_portfolios(self):
        crud = self.modules.get('crud_operations')
        if not crud:
            print("❌ CRUD module not available")
            return
        
        print("\n📊 Calling CRUD Module: get_all_portfolios()")
        print("="*60)
        portfolios = crud.get_all_portfolios(self.db_config)
        if portfolios:
            for p in portfolios:
                print(f"ID: {p[0]:2d} | {p[1]}")
            print("="*60)
            print(f"Total: {len(portfolios)} portfolios")
    
    def view_portfolio_details(self, portfolio_id):
        crud = self.modules.get('crud_operations')
        if not crud:
            print("❌ CRUD module not available")
            return
        
        print(f"\n📊 Calling CRUD Module: get_portfolio_details({portfolio_id})")
        print("="*60)
        details = crud.get_portfolio_details(portfolio_id, self.db_config)
        if details:
            print(f"Name: {details[0][1]}")
            print(f"Description: {details[0][2]}")
            
            # Get ETFs
            conn = self.get_connection()
            df = pd.read_sql(f"""
            SELECT pe.ticker, e.name, pe.weight
            FROM portfolio_etfs pe
            JOIN etfs e ON pe.ticker = e.ticker
            WHERE pe.portfolio_id = {portfolio_id}
            ORDER BY pe.weight DESC
            """, conn)
            conn.close()
            
            print("\nETFs:")
            for _, row in df.iterrows():
                print(f"  {row['ticker']:6s} {row['weight']:5.1f}%  {row['name']}")
            print("="*60)
    
    # ================================================================
    # ETF FUNCTIONS
    # ================================================================
    
    def view_etfs(self):
        crud = self.modules.get('crud_operations')
        if not crud:
            print("❌ CRUD module not available")
            return
        
        print("\n📊 Calling CRUD Module: get_all_etfs()")
        print("="*60)
        etfs = crud.get_all_etfs(self.db_config)
        if etfs:
            for etf in etfs[:20]:  # Show first 20
                print(f"{etf[0]:6s} {etf[1]:50s} {etf[2]}")
            print("...")
            print("="*60)
            print(f"Total: {len(etfs)} ETFs (showing first 20)")
    
    # ================================================================
    # PRICE FUNCTIONS
    # ================================================================
    
    def view_prices(self, ticker, limit):
        print(f"\n💹 Price Data: {ticker} (Latest {limit} days)")
        print("="*60)
        try:
            conn = self.get_connection()
            df = pd.read_sql(f"""
            SELECT date, close, volume
            FROM daily_prices
            WHERE ticker = '{ticker}'
            ORDER BY date DESC
            LIMIT {limit}
            """, conn)
            conn.close()
            
            if df.empty:
                print(f"❌ No data for {ticker}")
            else:
                for _, row in df.iterrows():
                    print(f"{row['date']}  ${row['close']:8.2f}  Vol: {row['volume']:,}")
                print("="*60)
        except Exception as e:
            print(f"❌ Error: {e}")
    
    # ================================================================
    # BACKTEST FUNCTIONS
    # ================================================================
    
    def run_backtest(self, portfolio_id, strategy, start_date, end_date, capital):
        backtest = self.modules.get('backtesting_engine')
        if not backtest:
            print("❌ Backtesting module not available")
            return
        
        print(f"\n🔬 Calling Backtesting Module: run_backtest()")
        print("="*60)
        print("Running... Please wait...")
        
        try:
            backtest_id = backtest.run_backtest(
                portfolio_id=portfolio_id,
                start_date=start_date,
                end_date=end_date,
                initial_capital=capital,
                strategy=strategy,
                db_config=self.db_config
            )
            
            if backtest_id:
                print(f"\n✅ Backtest completed! ID: {backtest_id}")
                
                conn = self.get_connection()
                df = pd.read_sql(f"""
                SELECT final_value, total_return
                FROM backtests WHERE backtest_id = {backtest_id}
                """, conn)
                conn.close()
                
                print(f"Final Value: ${df['final_value'].values[0]:,.2f}")
                print(f"Total Return: {df['total_return'].values[0]:.2f}%")
            else:
                print("❌ Backtest failed")
        except Exception as e:
            print(f"❌ Error: {e}")
    
    def view_backtest_history(self):
        print("\n📈 Backtest History (Latest 10)")
        print("="*60)
        try:
            conn = self.get_connection()
            df = pd.read_sql("""
            SELECT b.backtest_id, p.name, b.strategy_type, b.total_return
            FROM backtests b
            JOIN portfolios p ON b.portfolio_id = p.portfolio_id
            ORDER BY b.created_at DESC
            LIMIT 10
            """, conn)
            conn.close()
            
            if df.empty:
                print("⚠️  No backtests found")
            else:
                for _, row in df.iterrows():
                    print(f"ID: {row['backtest_id']:3d} | {row['name']:20s} | {row['strategy_type']:15s} | {row['total_return']:6.2f}%")
                print("="*60)
        except Exception as e:
            print(f"❌ Error: {e}")
    
    # ================================================================
    # ANALYTICS FUNCTIONS
    # ================================================================
    
    def run_analytics(self, insight_type, portfolio_ids):
        analytics = self.modules.get('analytics')
        if not analytics:
            print("❌ Analytics module not available")
            return
        
        print(f"\n📈 Calling Analytics Module: generate_insight{insight_type}()")
        print("="*60)
        print("Running... This may take a few minutes...\n")
        
        try:
            if insight_type == 1:
                analytics.generate_insight1(
                    portfolio_ids=portfolio_ids,
                    start_date='2020-01-01',
                    end_date='2024-12-31',
                    benchmark='SPY',
                    db_config=self.db_config
                )
                print("✅ Analysis completed!")
                print("📄 Report: insight1_risk_adjusted_report.txt")
            
            elif insight_type == 2:
                analytics.generate_insight2(
                    portfolio_id=portfolio_ids[0],
                    start_date='2020-01-01',
                    end_date='2024-12-31',
                    initial_capital=100000.0,
                    db_config=self.db_config
                )
                print("✅ Analysis completed!")
                print("📄 Report: insight2_rebalancing_analysis.txt")
            
            elif insight_type == 3:
                analytics.generate_insight3(
                    portfolio_id=portfolio_ids[0],
                    total_capital=100000.0,
                    investment_months=12,
                    start_date='2020-01-01',
                    end_date='2024-12-31',
                    db_config=self.db_config
                )
                print("✅ Analysis completed!")
                print("📄 Report: insight3_dca_vs_lumpsum.txt")
        
        except Exception as e:
            print(f"❌ Error: {e}")
    
    # ================================================================
    # SYSTEM STATS
    # ================================================================
    
    def show_stats(self):
        print("\n📊 SYSTEM STATISTICS")
        print("="*60)
        try:
            conn = self.get_connection()
            cursor = conn.cursor()
            
            tables = ['etfs', 'portfolios', 'daily_prices', 'backtests']
            for table in tables:
                cursor.execute(f"SELECT COUNT(*) FROM {table}")
                count = cursor.fetchone()[0]
                print(f"{table:20s}: {count:,}")
            
            cursor.close()
            conn.close()
            print("="*60)
        except Exception as e:
            print(f"❌ Error: {e}")

# ===================================================================
# CREATE & INITIALIZE SYSTEM
# ===================================================================

system = IntegratedSystem(DB_CONFIG, PROJECT_DIR)

if system.initialize():
    print("\n✅ พร้อมใช้งาน! Run Cell 2 เพื่อเริ่ม Main Controller")
else:
    print("\n❌ Initialization failed!")

# Cell 2: Run Main Controller

**Run cell นี้แล้วใช้ Menu ไปเรื่อยๆ จนกว่าจะเลือก 0 (Exit)**

In [ ]:
# ===================================================================
# MAIN CONTROLLER LOOP
# ===================================================================

def main_controller():
    """Main Controller - Interactive Menu Loop"""
    
    system.running = True
    
    while system.running:
        # Clear and show menu
        clear_output(wait=True)
        
        print("\n" + "="*80)
        print("  ETF PORTFOLIO BACKTESTING SYSTEM - Main Controller")
        print("="*80)
        print("\n📋 MAIN MENU:")
        print("-" * 60)
        print("  1. View All Portfolios")
        print("  2. View Portfolio Details")
        print("  3. View All ETFs")
        print("  4. View Price Data")
        print("  5. Run Backtest")
        print("  6. View Backtest History")
        print("  7. Run Analytics")
        print("  8. System Statistics")
        print("  0. Exit")
        print("-" * 60)
        
        # Get user choice
        choice = input("\nEnter choice: ").strip()
        
        # Handle choice
        if choice == '1':
            system.view_portfolios()
            input("\nPress Enter to continue...")
        
        elif choice == '2':
            try:
                portfolio_id = int(input("Enter Portfolio ID: "))
                system.view_portfolio_details(portfolio_id)
            except ValueError:
                print("❌ Invalid input")
            input("\nPress Enter to continue...")
        
        elif choice == '3':
            system.view_etfs()
            input("\nPress Enter to continue...")
        
        elif choice == '4':
            ticker = input("Enter Ticker (e.g., SPY): ").strip().upper()
            try:
                limit = int(input("Number of days (default 10): ") or "10")
                system.view_prices(ticker, limit)
            except ValueError:
                print("❌ Invalid input")
            input("\nPress Enter to continue...")
        
        elif choice == '5':
            try:
                portfolio_id = int(input("Portfolio ID: "))
                strategy = input("Strategy (buy_hold/rebalancing_monthly/dca): ").strip()
                start_date = input("Start Date (YYYY-MM-DD): ").strip()
                end_date = input("End Date (YYYY-MM-DD): ").strip()
                capital = float(input("Initial Capital (default 100000): ") or "100000")
                
                system.run_backtest(portfolio_id, strategy, start_date, end_date, capital)
            except ValueError:
                print("❌ Invalid input")
            input("\nPress Enter to continue...")
        
        elif choice == '6':
            system.view_backtest_history()
            input("\nPress Enter to continue...")
        
        elif choice == '7':
            print("\nAnalytics Options:")
            print("  1. Risk-Adjusted Performance")
            print("  2. Optimal Rebalancing Frequency")
            print("  3. DCA vs Lump Sum")
            
            try:
                insight = int(input("\nSelect insight (1-3): "))
                ids_input = input("Portfolio IDs (comma-separated): ").strip()
                portfolio_ids = [int(x.strip()) for x in ids_input.split(',')]
                
                system.run_analytics(insight, portfolio_ids)
            except ValueError:
                print("❌ Invalid input")
            input("\nPress Enter to continue...")
        
        elif choice == '8':
            system.show_stats()
            input("\nPress Enter to continue...")
        
        elif choice == '0':
            clear_output(wait=True)
            print("\n" + "="*80)
            print("👋 Thank you for using ETF Portfolio Backtesting System!")
            print("="*80)
            system.running = False
            break
        
        else:
            print("\n❌ Invalid choice! Please try again.")
            input("\nPress Enter to continue...")

# ===================================================================
# START MAIN CONTROLLER
# ===================================================================

print("\n🚀 Starting Main Controller...")
print("\nYou will see the menu in a moment.")
print("Use the menu to navigate the system.")
print("Select 0 to exit when done.\n")

input("Press Enter to start...")

main_controller()

---

# 📚 Summary

## ✅ การใช้งาน:

1. **Run Cell 1:** Setup & Load (ครั้งเดียว)
2. **Run Cell 2:** Main Controller (รันแล้วใช้ไปเรื่อยๆ)
3. **ใช้ Menu:** เลือกคำสั่งที่ต้องการ
4. **Exit:** เลือก 0 เมื่อเสร็จ

## 🎯 Features:

- ✅ Main Controller รันในลูป
- ✅ Menu แบบ interactive
- ✅ เรียก modules ตามความเหมาะสม
- ✅ แสดงผลลัพธ์ทันที
- ✅ กลับไป menu อัตโนมัติ
- ✅ Exit เมื่อต้องการ

## 🏗️ Architecture:

```
User Input → Main Controller → Subsystem Module → Database → Results → Menu
                ↑_____________________________________________________________|
                                    (Loop until Exit)
```

---

**Integrated System - Run ครั้งเดียว ใช้ได้ทั้งหมด!** 🚀

---